# Analyst Consensus (IAA)

Broker target prices, earnings forecasts and **research PDF links** for any SET stock — the data
behind the *Analyst Consensus* table on Settrade's quote page
(`https://www.settrade.com/th/equities/quote/GULF/analyst-consensus`).

The payload splits into the two tables this notebook works with:

| Table | Method | Rows |
|---|---|---|
| Aggregates | `stats_to_dataframe()` | `average`, `median`, `high`, `low` |
| Per broker | `to_dataframe()` | one per covering broker, with the PDF link |

Endpoints:
- `GET https://www.settrade.com/api/set-fund/consensus/stock/{symbol}/consensus`
- `GET https://www.settrade.com/api/set-fund/consensus/stock/overall?lang=&symbol=`

> This is the library's only `www.settrade.com` service. Cookies and the mandatory `Referer` are
> handled automatically — there is nothing extra to pass.

## Setup

In [1]:
# !pip install "settfex[dataframe]"

from settfex.services.set import (
    Stock,
    get_analyst_consensus,
    get_analyst_consensus_dataframes,
    get_consensus_overall,
)
from settfex.utils.logging import setup_logger

setup_logger(level="CRITICAL")  # quiet the per-request logs (incl. the intentional 500s in §7)

## 1. Fetch the consensus

`get_analyst_consensus` returns one `AnalystConsensus` holding both tables plus the year labels.

In [2]:
data = await get_analyst_consensus("GULF")

print(f"{data.symbol}: {data.count} covering brokers (coverage={data.has_coverage})")
print(f"Years: current={data.current_year}, next={data.next_year}, target={data.target_price_year}")
print(f"Most recent broker update: {data.latest_update}")

GULF: 16 covering brokers (coverage=True)
Years: current=2026, next=2027, target=2026
Most recent broker update: 2026-08-13 16:57:51+07:00


## 2. The aggregate rows — Average / Median / High / Low

Each aggregate row is a `ConsensusStatistic`, labelled by its `statistic` field.

⚠️ **Every column is aggregated independently** — the `high` row is *not* one broker's row. On
GULF, `high.target_price` came from one broker while `high.target_price_change` came from another.

In [3]:
for stat in data.statistics:
    print(
        f"{stat.statistic:<8} target={stat.target_price:>7}  "
        f"EPS={stat.current_year_eps:.2f}  P/E={stat.current_year_pe:.2f}"
    )

average  target=  78.75  EPS=2.42  P/E=27.00
median   target=   78.5  EPS=2.48  P/E=26.26
high     target=   91.0  EPS=2.67  P/E=32.18
low      target=   72.0  EPS=2.02  P/E=24.30


## 3. The two DataFrames

With the optional `dataframe` extra (`pip install "settfex[dataframe]"`), each table renders
straight to pandas. Note the column names are **year-agnostic** (`current_year_eps`, never
`eps_2026`) so downstream code does not break every January — the actual years ride along in
`df.attrs`.

In [4]:
stats_df = data.stats_to_dataframe()
stats_df.set_index("statistic")

,target_price,target_price_change,target_price_percent_change,current_year_eps,next_year_eps,current_year_net_profit,next_year_net_profit,current_year_pe,next_year_pe,current_year_pbv,next_year_pbv,current_year_div,next_year_div
statistic,,,,,,,,,,,,,
average,78.75,7.25,10.679362,2.421474,2.633319,36074.269918,39177.854667,26.999511,24.739911,2.779018,2.657589,2.207444,2.515557
median,78.50,7.25,10.679362,2.475000,2.640000,36457.000000,39022.000000,26.263591,24.621212,2.743774,2.713333,2.164677,2.592308
high,91.00,12.00,17.910448,2.674490,2.820000,39956.514510,42170.000000,32.178218,27.083333,3.023256,2.966981,2.953846,3.261538
low,72.00,2.50,3.448276,2.020000,2.400000,30133.000000,35879.000000,24.303699,23.049645,2.488973,2.326058,1.138462,1.307692


In [5]:
brokers_df = data.to_dataframe()
brokers_df[["broker_name", "analyst_name", "recommend", "target_price", "research_url"]].head(10)

,broker_name,analyst_name,recommend,target_price,research_url
0,ASPS,Tanya Udom,Buy,80.0,https://portal.settrade.com/brokerpage/Analyst...
1,BLS,Panjapon Taensricharoen,Buy,82.0,https://portal.settrade.com/brokerpage/Analyst...
2,CGSI,Rasmiman Sermprasert,Buy,75.0,None
3,DBSV,Duladeth BIK,Buy,79.0,None
4,FSSIA,Songklod Wongchai,Buy,76.0,https://portal.settrade.com/brokerpage/Analyst...
5,GLOBLEX,Suwat Sinsadok,Buy,82.0,https://portal.settrade.com/brokerpage/Analyst...
6,INVX,Chaiwat Arsirawichai,Outperform Market,78.0,None
7,KGI,"Wetid Tangjindakun, IAA, CISA",Outperform Market,78.0,https://portal.settrade.com/brokerpage/Analyst...
8,KS,Tan Chirasittikorn,Outperform Market,79.0,https://portal.settrade.com/brokerpage/Analyst...
9,KSS,Pannathat Laohatanasarn,Buy,77.0,https://portal.settrade.com/brokerpage/Analyst...


In [6]:
# Or fetch straight into both frames in one call
stats_df, brokers_df = await get_analyst_consensus_dataframes("CPALL")
print(f"stats {stats_df.shape} | brokers {brokers_df.shape} | attrs {brokers_df.attrs}")

stats (4, 14) | brokers (20, 19) | attrs {'symbol': 'CPALL', 'current_year': 2026, 'next_year': 2027, 'target_price_year': 2026, 'has_coverage': True}


## 4. Research PDF links

`research_urls` gives `(broker, url)` pairs for the brokers that published a report. Not every
broker does — on GULF only 9 of 16 did — so `last_research_url` is frequently `None`.

In [7]:
for broker, url in data.research_urls:
    print(f"{broker:<12} {url}")

print(f"\n{len(data.with_research)} of {data.count} brokers published a PDF")

ASPS         https://portal.settrade.com/brokerpage/AnalystConsensus/Research/ASPS_GULF_350186.pdf
BLS          https://portal.settrade.com/brokerpage/AnalystConsensus/Research/BLS_GULF_349749.pdf
FSSIA        https://portal.settrade.com/brokerpage/AnalystConsensus/Research/FSSIA_GULF_349283.pdf
GLOBLEX      https://portal.settrade.com/brokerpage/AnalystConsensus/Research/GLOBLEX_GULF_349079.pdf
KGI          https://portal.settrade.com/brokerpage/AnalystConsensus/Research/KGI_GULF_350307.pdf
KS           https://portal.settrade.com/brokerpage/AnalystConsensus/Research/KS_GULF_346083.pdf
KSS          https://portal.settrade.com/brokerpage/AnalystConsensus/Research/KSS_GULF_348198.pdf
LHS          https://portal.settrade.com/brokerpage/AnalystConsensus/Research/LHS_GULF_349113.pdf
YUANTA       https://portal.settrade.com/brokerpage/AnalystConsensus/Research/YUANTA_GULF_346567.pdf

9 of 16 brokers published a PDF


## 5. Buy / hold / sell summary

The second endpoint gives the recommendation counts shown above the table.

In [8]:
summary = await get_consensus_overall("GULF")
row = summary.get("GULF")

print(f"Last price:      {row.last_price}")
print(f"Coverage:        {row.total_coverage} brokers")
print(f"Buy/Hold/Sell:   {row.buy} / {row.hold} / {row.sell}")
print(f"Bullish:         {row.bullish}%")
print(f"Target (median): {row.median_target_price}   (average) {row.average_target_price}")

Last price:      64.5
Coverage:        16 brokers
Buy/Hold/Sell:   16 / 0 / 0
Bullish:         100.0%
Target (median): 78.5   (average) 78.75


## 6. Whole-market screener

Call the summary endpoint with **no symbol** and it returns every covered SET stock in a single
request — a ready-made consensus screener.

In [9]:
market = await get_consensus_overall()
market_df = market.to_dataframe()

print(f"{market.count} SET stocks have analyst coverage (as of {market.market_time})")
market_df.nlargest(10, "total_coverage")[
    ["symbol", "total_coverage", "buy", "hold", "sell", "average_target_price"]
]

252 SET stocks have analyst coverage (as of 2026-08-15 03:20:05.697719+07:00)


,symbol,total_coverage,buy,hold,sell,average_target_price
0,BDMS,21,18,3,0,24.249988
1,PTTEP,21,15,5,1,162.809496
2,BCH,20,17,1,2,11.900000
3,CPALL,20,20,0,0,61.384956
4,ITC,20,20,0,0,20.719013
5,OSP,20,15,3,1,19.616599
6,TU,20,17,3,0,14.926316
7,AOT,19,15,4,0,68.931053
8,DELTA,19,8,6,5,312.184211
9,PR9,19,19,0,0,22.473684


In [10]:
# Widely covered names where every analyst says Buy
widely_covered = market_df[market_df["total_coverage"] >= 15]
unanimous = widely_covered[widely_covered["hold"].eq(0) & widely_covered["sell"].eq(0)]
unanimous[["symbol", "total_coverage", "buy", "average_target_price"]]

,symbol,total_coverage,buy,average_target_price
3,CPALL,20,20,61.384956
4,ITC,20,20,20.719013
9,PR9,19,19,22.473684
21,CPN,17,17,78.105882
30,GULF,16,16,78.750000


## 7. Symbols without coverage

Two distinct cases, and they behave differently:

| Case | Behaviour | How to handle |
|---|---|---|
| Settrade has **no record** (valid stocks like `ABICO`, DRs, warrants) | `FetchError` with `status_code=500` — *not* a 404 | catch `FetchError` |
| Listed but **no broker covers it** (e.g. `TCC`) | HTTP 200, `count == 0`, aggregates zero-filled | check `has_coverage` |

Those zeros are Settrade's placeholder, kept verbatim — `has_coverage` is the guard.

In [11]:
from settfex.exceptions import FetchError

for symbol in ["GULF", "TCC", "ABICO"]:
    try:
        result = await get_analyst_consensus(symbol)
    except FetchError as exc:
        print(f"{symbol:<8} no consensus record (HTTP {exc.status_code})")
        continue
    if not result.has_coverage:
        print(
            f"{symbol:<8} listed, but no broker covers it "
            f"(average.target_price={result.average.target_price} <- placeholder)"
        )
        continue
    print(f"{symbol:<8} {result.count} brokers, median target {result.median.target_price}")

GULF     16 brokers, median target 78.5


TCC      listed, but no broker covers it (average.target_price=0.0 <- placeholder)


ABICO    no consensus record (HTTP 500)


## 8. Using the unified `Stock` class

`Stock.get_analyst_consensus()` caches on the instance (the data moves at most once a day);
`get_consensus_overall()` does not, because it carries a live `last_price`.

In [12]:
stock = Stock("PTT")

consensus = await stock.get_analyst_consensus()
overall = await stock.get_consensus_overall()

print(f"{stock.symbol}: {consensus.count} brokers, consensus = {overall.get('PTT').recommend_type}")

# Look up a single broker's view, case-insensitively
row = consensus.broker("asps")
print(row.analyst_name, row.recommend, row.target_price if row else "not covered by ASPS")

PTT: 16 brokers, consensus = buy
Nalinrat Kittikumpolrat Buy 47.99586


## Summary

- `get_analyst_consensus(symbol)` → aggregates + broker rows + research PDF links
- `stats_to_dataframe()` / `to_dataframe()` → the two DataFrames
- `get_analyst_consensus_dataframes(symbol)` → both in one call, as `(stats, brokers)`
- `get_consensus_overall(symbol)` → buy/hold/sell counts; **omit the symbol for the whole market**
- Check **`has_coverage`** before trusting the aggregates, and catch **`FetchError`** (HTTP 500)
  for symbols Settrade has no record of
- The table endpoint has **no language dimension**; only the summary endpoint takes `lang`

📖 Full documentation: [`docs/settfex/services/set/analyst_consensus.md`](../../docs/settfex/services/set/analyst_consensus.md)